# CS4152 — Class Activity 1
## Building and Training Neural Networks with Keras and TensorFlow

**Course:** Deep Learning and Neural Networks  
**Submission date in activity sheet:** 22 October 2026  
**Group members:** Zahra Abbas (S2024266095) Maimoona Anwar (S2024266056).




In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    r2_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score, recall_score
)
from sklearn.datasets import make_blobs
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATA_DIR = Path("Data")
DATA_DIR.mkdir(exist_ok=True)
print("TensorFlow:", tf.__version__)
print("Data folder:", DATA_DIR.resolve())


# Task 1 — Regression: NYC Taxi Fare Prediction




In [ ]:
taxi_path = DATA_DIR / "taxi-fares.csv"
if not taxi_path.exists():
    print("Dataset not found:", taxi_path)
    print("Download the NYC taxi-fares.csv dataset required by the activity and place it in Data/.")
else:
    taxi = pd.read_csv(taxi_path, parse_dates=["pickup_datetime"])
    print("Original shape:", taxi.shape)
    display(taxi.head())


In [ ]:
if taxi_path.exists():
    # Keep single-passenger trips.
    taxi = taxi[taxi["passenger_count"] == 1].copy()

    # Extract day and hour from pickup timestamp.
    taxi["day_of_week"] = taxi["pickup_datetime"].dt.weekday
    taxi["pickup_time"] = taxi["pickup_datetime"].dt.hour

    # Approximate straight-line distance in miles, using the class reading's factors.
    x_dist = (taxi["dropoff_longitude"] - taxi["pickup_longitude"]) * 54.6
    y_dist = (taxi["dropoff_latitude"] - taxi["pickup_latitude"]) * 69.0
    taxi["distance"] = np.sqrt(x_dist**2 + y_dist**2)

    # Remove columns not used as model features.
    taxi.drop(
        ["key", "passenger_count", "pickup_datetime", "pickup_longitude",
         "pickup_latitude", "dropoff_longitude", "dropoff_latitude"],
        axis=1, inplace=True, errors="ignore"
    )

    # Remove outliers as required by the activity.
    taxi = taxi[
        taxi["distance"].between(1.0, 10.0, inclusive="neither")
        & taxi["fare_amount"].between(0.0, 50.0, inclusive="neither")
    ].dropna()

    print("Shape after preprocessing:", taxi.shape)
    display(taxi.head())
    print("Preprocessing: single passenger only; extracted weekday/hour; estimated distance; "
          "removed unused columns, missing rows, distances outside (1,10) miles, and fares outside (0,50).")


In [ ]:
if taxi_path.exists():
    X_taxi = taxi[["day_of_week", "pickup_time", "distance"]].astype("float32")
    y_taxi = taxi["fare_amount"].astype("float32")

    taxi_model = Sequential([
        Input(shape=(3,)),
        Dense(512, activation="relu"),
        Dense(512, activation="relu"),
        Dense(1)  # Linear output is appropriate for a continuous fare value.
    ])
    taxi_model.compile(optimizer="adam", loss="mae", metrics=["mae"])
    taxi_model.summary()

    taxi_history = taxi_model.fit(
        X_taxi, y_taxi, epochs=100, batch_size=100,
        validation_split=0.2, verbose=1
    )


In [ ]:
if taxi_path.exists():
    plt.figure(figsize=(9, 5))
    plt.plot(taxi_history.history["mae"], label="Training MAE")
    plt.plot(taxi_history.history["val_mae"], label="Validation MAE")
    plt.title("Taxi Fare: Training and Validation MAE")
    plt.xlabel("Epoch")
    plt.ylabel("MAE ($)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # R² on the available rows is descriptive; a held-out test set is preferable
    # for an unbiased final estimate. This mirrors the class reading's example.
    taxi_predictions = taxi_model.predict(X_taxi, verbose=0).ravel()
    print("R² score on all preprocessed rows:", round(r2_score(y_taxi, taxi_predictions), 4))
    print("Final training MAE:", round(taxi_history.history["mae"][-1], 4))
    print("Final validation MAE:", round(taxi_history.history["val_mae"][-1], 4))


In [ ]:
if taxi_path.exists():
    friday = taxi_model.predict(np.array([[4, 17, 2.0]], dtype="float32"), verbose=0)[0, 0]
    saturday = taxi_model.predict(np.array([[5, 17, 2.0]], dtype="float32"), verbose=0)[0, 0]
    print(f"Friday, 5 PM, 2 miles: ${friday:.2f}")
    print(f"Saturday, 5 PM, 2 miles: ${saturday:.2f}")
    print(f"Saturday minus Friday: ${saturday-friday:.2f}")
    print("Interpretation: the model learned associations in this dataset. A small difference is "
          "plausible; the model has no explicit traffic or surge-pricing feature, so do not assume "
          "the weekday effect is causal.")


### Task 1 experiment: hidden-layer width and batch size




In [ ]:
if taxi_path.exists():
    Xtr, Xval, ytr, yval = train_test_split(
        X_taxi, y_taxi, test_size=0.2, random_state=SEED
    )

    def build_taxi_model(width):
        m = Sequential([
            Input(shape=(3,)),
            Dense(width, activation="relu"),
            Dense(width, activation="relu"),
            Dense(1)
        ])
        m.compile(optimizer="adam", loss="mae", metrics=["mae"])
        return m

    experiment_rows = []
    # Activity asks to vary width (128, 256, 512, 1024) and batch size.
    for width in [128, 256, 512, 1024]:
        for batch in [32, 100, 256]:
            tf.keras.backend.clear_session()
            tf.random.set_seed(SEED)
            m = build_taxi_model(width)
            start = time.time()
            h = m.fit(
                Xtr, ytr, validation_data=(Xval, yval),
                epochs=30, batch_size=batch, verbose=0
            )
            elapsed = time.time() - start
            experiment_rows.append({
                "hidden_width_each_layer": width,
                "batch_size": batch,
                "epochs": len(h.history["loss"]),
                "final_train_MAE": h.history["mae"][-1],
                "final_validation_MAE": h.history["val_mae"][-1],
                "training_time_seconds": elapsed
            })
    taxi_experiments = pd.DataFrame(experiment_rows).sort_values("final_validation_MAE")
    display(taxi_experiments)
    print("Note: experiments use 30 epochs to make comparison manageable; the main required model uses 100.")


# Task 2 — Binary Classification: Credit Card Fraud Detection




In [ ]:
fraud_path = DATA_DIR / "creditcard.csv"
if not fraud_path.exists():
    print("Dataset not found:", fraud_path)
    print("Place the creditcard.csv file from the activity dataset in Data/.")
else:
    fraud = pd.read_csv(fraud_path)
    print("Shape:", fraud.shape)
    display(fraud.head())
    print(fraud["Class"].value_counts())
    print("Class percentages:")
    display((fraud["Class"].value_counts(normalize=True) * 100).rename("percentage").to_frame())


In [ ]:
if fraud_path.exists():
    X_fraud = fraud.drop(["Time", "Class"], axis=1)
    y_fraud = fraud["Class"].astype("int32")

    X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
        X_fraud, y_fraud, test_size=0.2, stratify=y_fraud, random_state=SEED
    )
    print("Train class proportions:")
    display(y_train_f.value_counts(normalize=True).rename("proportion").to_frame())
    print("Test class proportions:")
    display(y_test_f.value_counts(normalize=True).rename("proportion").to_frame())

    fraud_model = Sequential([
        Input(shape=(29,)),
        Dense(128, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    fraud_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    fraud_model.summary()
    fraud_history = fraud_model.fit(
        X_train_f, y_train_f, validation_data=(X_test_f, y_test_f),
        epochs=10, batch_size=100, verbose=1
    )


In [ ]:
if fraud_path.exists():
    plt.figure(figsize=(8, 5))
    plt.plot(fraud_history.history["accuracy"], label="Training accuracy")
    plt.plot(fraud_history.history["val_accuracy"], label="Validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Fraud Model Accuracy")
    plt.legend()
    plt.tight_layout()
    plt.show()

    fraud_prob = fraud_model.predict(X_test_f, verbose=0).ravel()
    fraud_pred = (fraud_prob >= 0.5).astype(int)
    cm_fraud = confusion_matrix(y_test_f, fraud_pred, labels=[0, 1])
    ConfusionMatrixDisplay(cm_fraud, display_labels=["Legitimate (0)", "Fraud (1)"]).plot()
    plt.title("Fraud Detection — Without Class Weights")
    plt.tight_layout()
    plt.show()

    tn, fp, fn, tp = cm_fraud.ravel()
    tpr = tp / (tp + fn) if tp + fn else 0  # True positive rate / fraud recall
    fpr = fp / (fp + tn) if fp + tn else 0  # False positive rate
    print(f"Accuracy: {accuracy_score(y_test_f, fraud_pred):.5f}")
    print(f"True positive rate (fraud recall): {tpr:.5f}")
    print(f"False positive rate: {fpr:.5f}")
    print(classification_report(y_test_f, fraud_pred, target_names=["Legitimate", "Fraud"], zero_division=0))


In [ ]:
if fraud_path.exists():
    weighted_fraud_model = Sequential([
        Input(shape=(29,)),
        Dense(128, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    weighted_fraud_model.compile(
        optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"]
    )
    weighted_history = weighted_fraud_model.fit(
        X_train_f, y_train_f, validation_data=(X_test_f, y_test_f),
        epochs=10, batch_size=100, class_weight={0: 1.0, 1: 0.01}, verbose=1
    )

    weighted_prob = weighted_fraud_model.predict(X_test_f, verbose=0).ravel()
    weighted_pred = (weighted_prob >= 0.5).astype(int)
    cm_weighted = confusion_matrix(y_test_f, weighted_pred, labels=[0, 1])
    ConfusionMatrixDisplay(cm_weighted, display_labels=["Legitimate (0)", "Fraud (1)"]).plot()
    plt.title("Fraud Detection — Class Weights {0: 1.0, 1: 0.01}")
    plt.tight_layout()
    plt.show()

    def fraud_metrics(y_true, y_hat, label):
        cm = confusion_matrix(y_true, y_hat, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        return {
            "model": label,
            "accuracy": accuracy_score(y_true, y_hat),
            "fraud_true_positive_rate": tp/(tp+fn) if tp+fn else 0,
            "false_positive_rate": fp/(fp+tn) if fp+tn else 0,
            "false_negatives_missed_fraud": fn,
            "false_positives_legitimate_flagged": fp
        }
    display(pd.DataFrame([
        fraud_metrics(y_test_f, fraud_pred, "Without class weights"),
        fraud_metrics(y_test_f, weighted_pred, "Class weights 0:1.0, 1:0.01")
    ]))


# Task 3 — Multiclass Classification: Nonlinear Four-Class Data



In [ ]:
# Create a nonlinear four-class dataset: each class occupies a different radial band.
rng = np.random.default_rng(SEED)
n_per_class = 500
angles = rng.uniform(0, 2*np.pi, size=4*n_per_class)
labels_4 = np.repeat(np.arange(4), n_per_class)
rng.shuffle(labels_4)
# Each label has a different radius, with noise; the class boundaries are nonlinear.
radii = np.array([0.45, 0.95, 1.45, 1.95])[labels_4] + rng.normal(0, 0.10, size=4*n_per_class)
X_4 = np.column_stack([radii*np.cos(angles), radii*np.sin(angles)]).astype("float32")
y_4 = labels_4.astype("int32")

X_train_4, X_val_4, y_train_4, y_val_4 = train_test_split(
    X_4, y_4, test_size=0.2, random_state=SEED, stratify=y_4
)
plt.figure(figsize=(7, 6))
plt.scatter(X_4[:, 0], X_4[:, 1], c=y_4, s=10)
plt.title("Generated Nonlinear Dataset — Four Classes")
plt.xlabel("x")
plt.ylabel("y")
plt.show()


In [ ]:
multi_model = Sequential([
    Input(shape=(2,)),
    Dense(128, activation="relu"),
    Dense(4, activation="softmax")
])
multi_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
multi_model.summary()
multi_history = multi_model.fit(
    X_train_4, y_train_4, epochs=40, batch_size=10,
    validation_data=(X_val_4, y_val_4), verbose=1
)


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(multi_history.history["accuracy"], label="Training accuracy")
plt.plot(multi_history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Four-Class Model Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

point = np.array([[0.2, 0.8]], dtype="float32")
probabilities = multi_model.predict(point, verbose=0)[0]
predicted_class = int(np.argmax(probabilities))
print("Point:", point[0])
print("Probabilities for classes 0, 1, 2, 3:", np.round(probabilities, 4))
print("Predicted class:", predicted_class)
print("Softmax probabilities sum to:", probabilities.sum())


In [ ]:
# Decision boundary: predict every point in a 2D grid.
xx, yy = np.meshgrid(
    np.linspace(X_4[:, 0].min()-0.2, X_4[:, 0].max()+0.2, 250),
    np.linspace(X_4[:, 1].min()-0.2, X_4[:, 1].max()+0.2, 250)
)
grid = np.c_[xx.ravel(), yy.ravel()].astype("float32")
grid_pred = np.argmax(multi_model.predict(grid, batch_size=2048, verbose=0), axis=1).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, grid_pred, alpha=0.30, levels=np.arange(-0.5, 4.5, 1))
plt.scatter(X_val_4[:, 0], X_val_4[:, 1], c=y_val_4, s=10, edgecolors="black", linewidths=0.2)
plt.title("Decision Boundaries and Validation Points")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

y_pred_4 = np.argmax(multi_model.predict(X_val_4, verbose=0), axis=1)
cm_4 = confusion_matrix(y_val_4, y_pred_4, labels=[0, 1, 2, 3])
ConfusionMatrixDisplay(cm_4, display_labels=[0, 1, 2, 3]).plot()
plt.title("Four-Class Confusion Matrix")
plt.tight_layout()
plt.show()
print(classification_report(y_val_4, y_pred_4, labels=[0,1,2,3], zero_division=0))


# Task 4 — Regularization with Dropout


In [ ]:
def build_dropout_model(rate=None):
    layers = [Input(shape=(2,)), Dense(128, activation="relu")]
    if rate is not None and rate > 0:
        layers.append(Dropout(rate))
    layers.append(Dense(4, activation="softmax"))
    m = Sequential(layers)
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

dropout_rows = []
dropout_histories = {}
for rate in [None, 0.1, 0.2, 0.4]:
    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)
    label = "No dropout" if rate is None else f"Dropout {rate}"
    m = build_dropout_model(rate)
    h = m.fit(
        X_train_4, y_train_4, epochs=40, batch_size=10,
        validation_data=(X_val_4, y_val_4), verbose=0
    )
    dropout_histories[label] = h
    dropout_rows.append({
        "configuration": label,
        "final_train_accuracy": h.history["accuracy"][-1],
        "final_validation_accuracy": h.history["val_accuracy"][-1],
        "train_validation_gap": h.history["accuracy"][-1] - h.history["val_accuracy"][-1]
    })

dropout_comparison = pd.DataFrame(dropout_rows)
display(dropout_comparison)


In [ ]:
plt.figure(figsize=(9, 5))
for label, h in dropout_histories.items():
    plt.plot(h.history["val_accuracy"], label=label)
plt.xlabel("Epoch")
plt.ylabel("Validation accuracy")
plt.title("Validation Accuracy: Dropout Comparison")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 5))
for label, h in dropout_histories.items():
    plt.plot(h.history["accuracy"], label=label)
plt.xlabel("Epoch")
plt.ylabel("Training accuracy")
plt.title("Training Accuracy: Dropout Comparison")
plt.legend()
plt.tight_layout()
plt.show()


# Task 5 — Save, Load, and Callbacks




In [ ]:
# Callbacks demonstration on the four-class model (runs independently of taxi data).
callback_model = Sequential([
    Input(shape=(2,)),
    Dense(128, activation="relu"),
    Dense(4, activation="softmax")
])
callback_model.compile(
    optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ModelCheckpoint("best_multiclass_model.keras", monitor="val_loss",
                    save_best_only=True, verbose=1)
]
callback_history = callback_model.fit(
    X_train_4, y_train_4, validation_data=(X_val_4, y_val_4),
    epochs=100, batch_size=10, callbacks=callbacks, verbose=1
)
print("Epochs actually run:", len(callback_history.history["loss"]))
print("Best validation loss:", min(callback_history.history["val_loss"]))


In [ ]:
# Save/reload the taxi model only if the required taxi dataset was available and trained.
if taxi_path.exists():
    taxi_model.save("best_taxi_model.h5")

    saved_model_dir = Path("best_taxi_savedmodel")
    if hasattr(taxi_model, "export"):
        taxi_model.export(str(saved_model_dir))  # Keras 3 SavedModel export
        print("Saved TensorFlow SavedModel to:", saved_model_dir)
    else:
        taxi_model.save(str(saved_model_dir), save_format="tf")
        print("Saved TensorFlow SavedModel to:", saved_model_dir)

    reloaded_taxi = load_model("best_taxi_model.h5")
    sample = np.array([[4, 17, 2.0], [5, 17, 2.0]], dtype="float32")
    original_predictions = taxi_model.predict(sample, verbose=0)
    reloaded_predictions = reloaded_taxi.predict(sample, verbose=0)
    print("Original predictions:", original_predictions.ravel())
    print("Reloaded predictions:", reloaded_predictions.ravel())
    print("Predictions match within tolerance:",
          np.allclose(original_predictions, reloaded_predictions, atol=1e-5))
else:
    print("Taxi model save/reload skipped: run Task 1 after adding Data/taxi-fares.csv.")
